# Enrichissement World Bank — Données socio-économiques

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
from pathlib import Path

# Chemins
DATA_RAW      = Path('../data/raw')
FAO_COMPLET   = DATA_RAW / 'FAO_complet_1961_2023.csv'
OUTPUT_WB     = DATA_RAW / 'worldbank_socioeco.csv'

# Indicateurs World Bank à récupérer
INDICATEURS = {
    'NY.GDP.PCAP.CD'   : 'pib_per_capita',
    'SP.URB.TOTL.IN.ZS': 'taux_urbanisation',
    'SP.POP.TOTL'      : 'population',
    'AG.LND.AGRI.ZS'   : 'surface_agricole',
}

# Période
ANNEE_DEBUT = 1961
ANNEE_FIN   = 2023

# URL World Bank REST API v2
WB_API_BASE = "https://api.worldbank.org/v2"

print("✅ Imports OK")
print(f"📊 Indicateurs : {list(INDICATEURS.values())}")
print(f"📅 Période     : {ANNEE_DEBUT}–{ANNEE_FIN}")

---
## 2. Mapping pays FAO → codes ISO3 World Bank

Le World Bank utilise les codes **ISO 3166-1 alpha-3** (ex: `FRA`, `USA`).  
FAO.csv utilise ses propres noms de pays.  
On utilise la liste officielle World Bank pour faire la correspondance.

In [ ]:
# 2.1 — Charger la liste des pays du fichier FAO complet
print("📂 Chargement FAO_complet...")
df_fao = pd.read_csv(FAO_COMPLET, low_memory=False)
pays_fao = sorted(df_fao['Area'].unique())
print(f"  {len(pays_fao)} pays uniques dans FAO_complet_1961_2023.csv")

In [ ]:
# 2.2 — Récupérer la liste officielle des pays World Bank (avec codes ISO3)
print("🌐 Récupération liste pays World Bank...")

resp = requests.get(
    f"{WB_API_BASE}/country?format=json&per_page=500",
    timeout=30
)
resp.raise_for_status()
data_wb = resp.json()

# L'API retourne [metadata, [liste de pays]]
pays_wb_raw = data_wb[1]

# Garder uniquement les pays (pas les agrégats régionaux)
# Les agrégats ont capitalLevel = "Aggregates"
pays_wb = [
    {
        'iso3'   : p['id'],
        'iso2'   : p['iso2Code'].strip(),
        'nom_wb' : p['name'],
        'region' : p['region']['value'],
    }
    for p in pays_wb_raw
    if p['region']['value'] != 'Aggregates'
]

df_pays_wb = pd.DataFrame(pays_wb)
print(f"  ✅ {len(df_pays_wb)} pays individuels World Bank")
df_pays_wb.head()

In [ ]:
# 2.3 — Correspondance automatique FAO ↔ World Bank par nom de pays
# On normalise les noms (minuscules, sans accents) pour maximiser les matches

import unicodedata

def normaliser(s):
    """Normalise un nom de pays pour la comparaison : minuscules, sans accents."""
    s = s.lower().strip()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s

# Dictionnaire nom normalisé → iso3 (World Bank)
wb_norm = {normaliser(r['nom_wb']): r['iso3'] for _, r in df_pays_wb.iterrows()}

# Corrections manuelles pour les pays dont le nom diffère entre FAO et World Bank
CORRECTIONS_MANUELLES = {
    # FAO name                          : ISO3 World Bank
    'Bolivia (Plurinational State of)'  : 'BOL',
    'China, mainland'                   : 'CHN',
    'China, Hong Kong SAR'              : 'HKG',
    'China, Macao SAR'                  : 'MAC',
    'China, Taiwan Province of'         : None,   # pas dans WB
    "Côte d'Ivoire"                     : 'CIV',
    'Democratic Republic of the Congo'  : 'COD',
    "Democratic People's Republic of Korea": 'PRK',
    'Iran (Islamic Republic of)'        : 'IRN',
    "Lao People's Democratic Republic"  : 'LAO',
    'Micronesia (Federated States of)'  : 'FSM',
    'Republic of Korea'                 : 'KOR',
    'Republic of Moldova'               : 'MDA',
    'Russian Federation'                : 'RUS',
    'Syrian Arab Republic'              : 'SYR',
    'Tanzania, United Republic of'      : 'TZA',
    'United Kingdom of Great Britain and Northern Ireland': 'GBR',
    'United States of America'          : 'USA',
    'Venezuela (Bolivarian Republic of)': 'VEN',
    'Viet Nam'                          : 'VNM',
    'United Republic of Tanzania'       : 'TZA',
    'Czechia'                           : 'CZE',
    'Congo'                             : 'COG',
    'Eswatini'                          : 'SWZ',
    'North Macedonia'                   : 'MKD',
    'Timor-Leste'                       : 'TLS',
    'Cabo Verde'                        : 'CPV',
    'Kyrgyzstan'                        : 'KGZ',
    'Yemen'                             : 'YEM',
    'Ethiopia PDR'                      : 'ETH',
    'Sudan (former)'                    : None,   # entité historique
    'USSR'                              : None,   # entité historique
    'Yugoslavia SFR'                    : None,   # entité historique
    'Czechoslovakia'                    : None,   # entité historique
    'German Democratic Republic'        : None,   # entité historique
    'Germany, Federal Republic of'      : 'DEU',
}

# Construire le mapping FAO → ISO3
mapping_fao_iso3 = {}
non_trouves = []

for pays in pays_fao:
    if pays in CORRECTIONS_MANUELLES:
        mapping_fao_iso3[pays] = CORRECTIONS_MANUELLES[pays]
    elif normaliser(pays) in wb_norm:
        mapping_fao_iso3[pays] = wb_norm[normaliser(pays)]
    else:
        mapping_fao_iso3[pays] = None
        non_trouves.append(pays)

trouves   = sum(1 for v in mapping_fao_iso3.values() if v is not None)
manquants = [k for k, v in mapping_fao_iso3.items() if v is None]

print(f"✅ Pays mappés      : {trouves}/{len(pays_fao)}")
print(f"⚠️  Pays non mappés : {len(manquants)}")
if manquants:
    print("   (entités historiques ou territoires non reconnus par WB)")
    for p in sorted(manquants):
        print(f"   - {p}")

---
## 3. Récupération des données World Bank

On utilise l'API REST World Bank v2 :  
`GET /v2/country/{iso3}/indicator/{indicator}?format=json&date=1961:2023&per_page=1000`

Stratégie : **batch par indicateur** (1 requête pour tous les pays d'un coup via `all` + filtre ensuite).

In [ ]:
# 3.1 — Fonction de récupération d'un indicateur pour tous les pays
# L'API WB accepte jusqu'à ~150 pays en une seule requête via point-virgule

def fetch_wb_indicator(indicator_code, iso3_list, date_range="1961:2023", pause=0.5):
    """
    Récupère un indicateur World Bank pour une liste de pays ISO3.
    Retourne un DataFrame avec colonnes : iso3, year, value.
    """
    # L'API accepte 'all' pour tous les pays — plus efficace que batch par pays
    url = (
        f"{WB_API_BASE}/country/all/indicator/{indicator_code}"
        f"?format=json&date={date_range}&per_page=20000"
    )

    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    if len(data) < 2 or data[1] is None:
        print(f"  ⚠️  Aucune donnée pour {indicator_code}")
        return pd.DataFrame(columns=['iso3', 'year', 'value'])

    rows = []
    for record in data[1]:
        iso3  = record['countryiso3code']
        year  = int(record['date'])
        value = record['value']
        if iso3 in iso3_list:
            rows.append({'iso3': iso3, 'year': year, 'value': value})

    df = pd.DataFrame(rows)

    # Gérer la pagination si nécessaire
    total_pages = data[0].get('pages', 1)
    if total_pages > 1:
        print(f"  📄 Pagination : {total_pages} pages pour {indicator_code}")
        for page in range(2, total_pages + 1):
            url_page = url + f"&page={page}"
            resp_p   = requests.get(url_page, timeout=60)
            resp_p.raise_for_status()
            data_p = resp_p.json()
            if data_p[1]:
                for record in data_p[1]:
                    iso3  = record['countryiso3code']
                    year  = int(record['date'])
                    value = record['value']
                    if iso3 in iso3_list:
                        rows.append({'iso3': iso3, 'year': year, 'value': value})
            time.sleep(pause)
        df = pd.DataFrame(rows)

    time.sleep(pause)
    return df

print("✅ Fonction fetch_wb_indicator définie")

In [ ]:
# 3.2 — Récupération de tous les indicateurs
# Liste des ISO3 valides (pays FAO mappés vers WB)
iso3_valides = set(v for v in mapping_fao_iso3.values() if v is not None)
print(f"🌍 {len(iso3_valides)} pays avec code ISO3 valide")

frames_indicateurs = {}

for code, nom in INDICATEURS.items():
    print(f"\n⏳ [{nom}] {code}...")
    df_ind = fetch_wb_indicator(code, iso3_valides)
    df_ind = df_ind.rename(columns={'value': nom})
    frames_indicateurs[nom] = df_ind
    non_null = df_ind[nom].notna().sum()
    print(f"  ✅ {len(df_ind):,} lignes — {non_null:,} valeurs non nulles ({non_null/len(df_ind)*100:.0f}%)")

print("\n✅ Tous les indicateurs récupérés")

---
## 4. Construction du DataFrame final

On pivote : une ligne par `(iso3, year)` avec tous les indicateurs en colonnes.

In [ ]:
# 4.1 — Jointure de tous les indicateurs sur (iso3, year)
print("🔗 Jointure des indicateurs...")

df_wb = None
for nom, df_ind in frames_indicateurs.items():
    if df_wb is None:
        df_wb = df_ind[['iso3', 'year', nom]].copy()
    else:
        df_wb = df_wb.merge(df_ind[['iso3', 'year', nom]], on=['iso3', 'year'], how='outer')

df_wb = df_wb.sort_values(['iso3', 'year']).reset_index(drop=True)

print(f"  Lignes   : {len(df_wb):,}")
print(f"  Pays     : {df_wb['iso3'].nunique()}")
print(f"  Années   : {df_wb['year'].min()}–{df_wb['year'].max()}")
print(f"\nCouverture par indicateur :")
for col in INDICATEURS.values():
    n_non_null = df_wb[col].notna().sum()
    pct = n_non_null / len(df_wb) * 100
    print(f"  {col:<22} {n_non_null:>7,} non-null ({pct:.0f}%)")

df_wb.head()

In [ ]:
# 4.2 — Ajouter le nom FAO en colonne (pour faciliter les jointures ultérieures)
# Mapping inverse : iso3 → nom_fao (on prend le premier nom FAO qui correspond)
iso3_to_fao = {}
for nom_fao, iso3 in mapping_fao_iso3.items():
    if iso3 and iso3 not in iso3_to_fao:
        iso3_to_fao[iso3] = nom_fao

df_wb['nom_fao'] = df_wb['iso3'].map(iso3_to_fao)

# Ajouter la région World Bank
iso3_to_region = {r['iso3']: r['region'] for _, r in df_pays_wb.iterrows()}
df_wb['region_wb'] = df_wb['iso3'].map(iso3_to_region)

# Réordonner les colonnes
df_wb = df_wb[['iso3', 'nom_fao', 'region_wb', 'year'] + list(INDICATEURS.values())]

print(f"✅ Colonnes finales : {df_wb.columns.tolist()}")
df_wb.head(10)

---
## 5. Gestion des valeurs manquantes

Les données World Bank ont des lacunes, notamment :
- Années **avant 1990** peu couvertes pour les anciens pays soviétiques
- Pays **disparus** (USSR, Yougoslavie...) → pas de données WB
- Certains petits territoires ont des données très parcellaires

Stratégie : **interpolation linéaire** par pays pour les petits gaps, pas d'extrapolation hors plage.

In [ ]:
# 5.1 — Diagnostic des valeurs manquantes par pays et indicateur
print("🔍 Taux de complétion par indicateur et par pays :")

for col in INDICATEURS.values():
    completude = df_wb.groupby('iso3')[col].apply(lambda x: x.notna().mean())
    n_complet  = (completude == 1.0).sum()
    n_partiel  = ((completude > 0) & (completude < 1)).sum()
    n_vide     = (completude == 0).sum()
    print(f"\n  {col} :")
    print(f"    Complet  (100%) : {n_complet} pays")
    print(f"    Partiel  (>0%)  : {n_partiel} pays")
    print(f"    Vide     (0%)   : {n_vide} pays")
    if n_vide > 0:
        pays_vides = completude[completude == 0].index.tolist()[:10]
        print(f"    Exemples vides : {pays_vides}")

In [ ]:
# 5.2 — Interpolation linéaire pour les gaps internes (pas d'extrapolation)
print("🔧 Interpolation linéaire des gaps internes...")

df_wb_interp = df_wb.copy()

for col in INDICATEURS.values():
    avant = df_wb_interp[col].isna().sum()
    df_wb_interp[col] = (
        df_wb_interp
        .groupby('iso3')[col]
        .transform(lambda x: x.interpolate(method='linear', limit_area='inside'))
    )
    apres = df_wb_interp[col].isna().sum()
    print(f"  {col:<22} NaN avant={avant:>6,} → après={apres:>6,} (comblés: {avant-apres:,})")

print("\n✅ Interpolation terminée (gaps aux extrémités conservés comme NaN)")

---
## 6. Sauvegarde

In [ ]:
# 6.1 — Sauvegarder (version brute + version interpolée)
OUTPUT_WB_INTERP = DATA_RAW / 'worldbank_socioeco_interp.csv'

df_wb.to_csv(OUTPUT_WB, index=False)
df_wb_interp.to_csv(OUTPUT_WB_INTERP, index=False)

print("💾 Sauvegarde terminée :")
print(f"  ✅ {OUTPUT_WB.name}")
print(f"     → {len(df_wb):,} lignes, données brutes")
print(f"  ✅ {OUTPUT_WB_INTERP.name}")
print(f"     → {len(df_wb_interp):,} lignes, gaps internes interpolés")

---
## 7. Validation

In [ ]:
# 7.1 — Vérification des ordres de grandeur
# (test sur quelques pays de référence)
PAYS_REF = ['USA', 'FRA', 'IND', 'CHN', 'BRA']
ANNEES_REF = [1990, 2000, 2010, 2020]

print("🔍 Vérification des valeurs de référence :")
print("\n--- PIB par habitant (USD) ---")
pivot_pib = df_wb_interp[df_wb_interp['iso3'].isin(PAYS_REF) & df_wb_interp['year'].isin(ANNEES_REF)]\
    .pivot(index='iso3', columns='year', values='pib_per_capita')
print(pivot_pib.to_string())

print("\n--- Population totale ---")
pivot_pop = df_wb_interp[df_wb_interp['iso3'].isin(PAYS_REF) & df_wb_interp['year'].isin(ANNEES_REF)]\
    .pivot(index='iso3', columns='year', values='population')
print(pivot_pop.to_string())

In [ ]:
# 7.2 — Résumé de couverture par rapport à FAO_complet
print("📊 Couverture World Bank vs FAO :")

pays_fao_avec_iso3 = {k for k, v in mapping_fao_iso3.items() if v is not None}
pays_fao_sans_iso3 = {k for k, v in mapping_fao_iso3.items() if v is None}

pays_wb_dans_df = df_wb_interp['iso3'].nunique()

print(f"  Pays dans FAO_complet         : {len(pays_fao)}")
print(f"  Pays mappés vers ISO3 WB      : {len(pays_fao_avec_iso3)} ({len(pays_fao_avec_iso3)/len(pays_fao)*100:.0f}%)")
print(f"  Pays sans équivalent WB       : {len(pays_fao_sans_iso3)} (entités historiques)")
print(f"  Pays avec données WB dans CSV : {pays_wb_dans_df}")

print(f"\n  Pays FAO sans équivalent WB :")
for p in sorted(pays_fao_sans_iso3):
    print(f"    - {p}")

In [ ]:
# 7.3 — Résumé final
print("=" * 60)
print("✅ ENRICHISSEMENT WORLD BANK TERMINÉ")
print("=" * 60)
print(f"""
Fichiers produits :
  📄 {OUTPUT_WB.name}
     → Données brutes World Bank
     → {len(df_wb):,} lignes

  📄 {OUTPUT_WB_INTERP.name}
     → Gaps internes interpolés
     → {len(df_wb_interp):,} lignes
     → {df_wb_interp['iso3'].nunique()} pays
     → {df_wb_interp['year'].min()}–{df_wb_interp['year'].max()}

Indicateurs :
  • pib_per_capita     (NY.GDP.PCAP.CD)
  • taux_urbanisation  (SP.URB.TOTL.IN.ZS)
  • population         (SP.POP.TOTL)
  • surface_agricole   (AG.LND.AGRI.ZS)
""")